In [90]:
import pandas as pd

In [91]:
data = pd.read_csv('/projects/immunestatus/rheum/2025_rheum/rheumatology2018_as_h_data.txt', sep='\t')

In [92]:
data['sample_id'].nunique()

42

In [119]:
import pandas as pd
from pathlib import Path

def convert_to_airr_format(df, output_dir, default_locus="beta"):
    Path(output_dir).mkdir(parents=True, exist_ok=True)

    for sample_id, group in df.groupby("sample_id"):
        airr_df = pd.DataFrame({
            "count": group["duplicate_count"],
            "junction_aa": group["junction_aa"],
            "v_call": group["v_call"].apply(lambda x: x.split(',')[0]),
            "j_call": group["j_call"].apply(lambda x: x.split(',')[0]),
            "locus": default_locus
        })
        airr_df = airr_df[airr_df.junction_aa.str.isalpha()]
        out_path = Path(output_dir) / f"{sample_id}.tsv"
        airr_df.to_csv(out_path, sep='\t', index=False)

In [120]:
convert_to_airr_format(data, '/projects/immunestatus/rheum/airr_format')

In [95]:
meta1 = pd.read_csv('/projects/immunestatus/rheum/2025_rheum/rheumatology2018_metadata.txt', sep='\t')

In [96]:
meta1

,df_name,donor_id,b27_status,disease_status,source,fraction
0,as_Abd_PB_F,Abd,+,as,PB,bulk
1,as_Abr_PB_F,Abr,+,as,PB,bulk
2,as_Ash-110_PB_F_p0,Ash-110,+,as,PB,bulk
3,as_Ash-111_PB_F_p0,Ash-111,+,as,PB,bulk
4,as_Bal_PB_F,Bal,+,as,PB,bulk
5,as_Bel_PB_F,Bel,+,as,PB,bulk
6,as_Bost_PB_F,Bost,+,as,PB,bulk
7,as_Chaad_PB_F,Chaad,+,as,PB,bulk
8,as_Evst_PB_F,Evst,+,as,PB,bulk
9,as_Gar_PB_F,Gar,+,as,PB,bulk


In [97]:
b27_neg = pd.read_csv('/projects/immunestatus/rheum/2025_rheum/graft-DLI-donor_withMHCstatus_selected/Koc.txt', sep='\t')

In [98]:
b27_neg


,cloneCount,cloneFraction,nSeqCDR3,aaSeqCDR3,bestVGene,bestDGene,bestJGene,bestVHit,bestDHit,bestJHit,-positionOfVEndTrimmed,-positionOfDBeginTrimmed,-positionOfDEndTrimmed,-positionOfJBeginTrimmed
0,6475.0,0.038065,TGTGCCAGCAGCCCCCCCTCCGGGGGGCACGAGCAGTACTTC,CASSPPSGGHEQYF,TRBV7-8,TRBD1,TRBJ2-7,TRBV7-8*00,TRBD1*00,TRBJ2-7*00,12,13,19,28
1,1688.0,0.009923,TGCAGTGCTCGCCCCCGGGACAGAGCTTTTAACTATGGCTACACCTTC,CSARPRDRAFNYGYTF,TRBV20-1,TRBD1,TRBJ1-2,TRBV20-1*00,TRBD1*00,TRBJ1-2*00,9,13,23,29
2,1216.0,0.007148,TGTGCCAGCAGTCCAGAGGGGGTCGAGACCCAGTACTTC,CASSPEGVETQYF,TRBV12-3,TRBD1,TRBJ2-5,TRBV12-3*00,TRBD1*00,TRBJ2-5*00,12,16,22,24
3,944.0,0.005549,TGTGCCAGCAGTTACTCGGGAAGCCCGAACACCGGGGAGCTGTTTTTT,CASSYSGSPNTGELFF,TRBV6-2,NaN,TRBJ2-2,TRBV6-2*00,NaN,TRBJ2-2*00,17,-1,-1,25
4,773.0,0.004544,TGTGCCAGCAGCGTATATGGGGCGGTCCAAGAGACCCAGTACTTC,CASSVYGAVQETQYF,TRBV9,TRBD1,TRBJ2-5,TRBV9*00,TRBD1*00,TRBJ2-5*00,15,18,23,23
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
100572,1.0,0.000006,TGTGCCACCGTCTTC,CATVF,TRBV21-1,NaN,TRBJ2-7,TRBV21-1*00,NaN,TRBJ2-7*00,7,-1,-1,11
100573,1.0,0.000006,TGTGCCAGTACTTC,CA_YF,TRBV7-8,NaN,TRBJ2-7,TRBV7-8*00,NaN,TRBJ2-7*00,8,-1,-1,5
100574,1.0,0.000006,TGCGCCAGTACTTC,CA_YF,TRBV5-1,NaN,TRBJ2-7,TRBV5-1*00,NaN,TRBJ2-7*00,8,-1,-1,5
100575,1.0,0.000006,TGCGCCAGCATTTT,CA_HF,TRBV5-1,NaN,TRBJ1-5,TRBV5-1*00,NaN,TRBJ1-5*00,10,-1,-1,1


In [121]:
import pandas as pd
from pathlib import Path

def convert_from_metadata_and_raw(
    metadata_file: str,
    raw_data_dir: str,
    output_dir: str,
    default_locus: str = "beta",
):
    # Чтение метаданных
    meta = pd.read_csv(metadata_file, sep='\t')
    Path(output_dir).mkdir(parents=True, exist_ok=True)

    for _, row in meta.iterrows():
        file_path = Path(raw_data_dir) / (row['id'] + '.txt')
        if not file_path.exists():
            print(f"Файл не найден: {file_path}")
            continue
        print(file_path)

        # Чтение сырых данных
        try:
            df = pd.read_csv(file_path, sep='\t')
        except Exception as e:
            print(f"Ошибка при чтении {file_path}: {e}")
            continue
        
        try:
            # Преобразование в AIRR-формат
            airr_df = pd.DataFrame({
                "count": df["Read.count"],
                "junction_aa": df["CDR3.amino.acid.sequence"],
                "v_call": df["V.gene"].apply(lambda x: x.split(',')[0]),
                "j_call": df["J.gene"].apply(lambda x: x.split(',')[0]),
                "locus": default_locus
            })
        except Exception as e:
            try:
                airr_df = pd.DataFrame({
                    "count": df["count"],
                    "junction_aa": df["cdr3aa"],
                    "v_call": df["v"].apply(lambda x: x.split(',')[0]),
                    "j_call": df["j"].apply(lambda x: x.split(',')[0]),
                    "locus": default_locus
                })
            except Exception as e:
                airr_df = pd.DataFrame({
                    "count": df["cloneCount"],
                    "junction_aa": df["aaSeqCDR3"],
                    "v_call": df["bestVGene"].apply(lambda x: x.split(',')[0]),
                    "j_call": df["bestJGene"].apply(lambda x: x.split(',')[0]),
                    "locus": default_locus
                })

        output_file = Path(output_dir) / f"{row['id']}.tsv"
        airr_df = airr_df[airr_df.junction_aa.str.isalpha()]
        airr_df.to_csv(output_file, sep='\t', index=False)


In [122]:
convert_from_metadata_and_raw('/projects/immunestatus/rheum/2025_rheum/b27neg_LCFG/meta.txt', 
                              '/projects/immunestatus/rheum/2025_rheum/b27neg_LCFG',
                              '/projects/immunestatus/rheum/airr_format'
                              )

/projects/immunestatus/rheum/2025_rheum/b27neg_LCFG/TA_pre0.txt
/projects/immunestatus/rheum/2025_rheum/b27neg_LCFG/DL_0.txt
/projects/immunestatus/rheum/2025_rheum/b27neg_LCFG/YB14_pre0.txt
/projects/immunestatus/rheum/2025_rheum/b27neg_LCFG/TV_PB_F.txt
/projects/immunestatus/rheum/2025_rheum/b27neg_LCFG/KB_0.txt
/projects/immunestatus/rheum/2025_rheum/b27neg_LCFG/Azh_0.txt
/projects/immunestatus/rheum/2025_rheum/b27neg_LCFG/KM.txt
/projects/immunestatus/rheum/2025_rheum/b27neg_LCFG/hLP.txt


In [123]:
convert_from_metadata_and_raw('/projects/immunestatus/rheum/2025_rheum/graft-DLI-donor_withMHCstatus_selected/meta.txt', 
                              '/projects/immunestatus/rheum/2025_rheum/graft-DLI-donor_withMHCstatus_selected',
                              '/projects/immunestatus/rheum/airr_format'
                              )

/projects/immunestatus/rheum/2025_rheum/graft-DLI-donor_withMHCstatus_selected/Koc.txt
/projects/immunestatus/rheum/2025_rheum/graft-DLI-donor_withMHCstatus_selected/Make.txt
/projects/immunestatus/rheum/2025_rheum/graft-DLI-donor_withMHCstatus_selected/p144852.txt
/projects/immunestatus/rheum/2025_rheum/graft-DLI-donor_withMHCstatus_selected/p1004.txt
/projects/immunestatus/rheum/2025_rheum/graft-DLI-donor_withMHCstatus_selected/p112635.txt
/projects/immunestatus/rheum/2025_rheum/graft-DLI-donor_withMHCstatus_selected/p434.txt
/projects/immunestatus/rheum/2025_rheum/graft-DLI-donor_withMHCstatus_selected/p556.txt
/projects/immunestatus/rheum/2025_rheum/graft-DLI-donor_withMHCstatus_selected/p744.txt
/projects/immunestatus/rheum/2025_rheum/graft-DLI-donor_withMHCstatus_selected/p754.txt
/projects/immunestatus/rheum/2025_rheum/graft-DLI-donor_withMHCstatus_selected/p1005.txt
/projects/immunestatus/rheum/2025_rheum/graft-DLI-donor_withMHCstatus_selected/p1321.txt
/projects/immunestatus/r

In [102]:
meta2 = pd.read_csv(
    '/projects/immunestatus/rheum/2025_rheum/graft-DLI-donor_withMHCstatus_selected/meta.txt', 
                    sep='\t').drop(columns=['file_format', 'filename'])

In [103]:
meta3 = pd.read_csv('/projects/immunestatus/rheum/2025_rheum/b27neg_LCFG/meta.txt', sep='\t')

In [104]:
meta2.rename(columns={'id': 'donor_id', 'state': 'disease_status', 'b27status': 'b27'})

,donor_id,disease_status,b27,B2705,B27other,proj,sample_type,time_point
0,Koc,HD,pos,2705pos,neg,GCSF,PBMC,normal blood
1,Make,HD,pos,2705pos,neg,ch45,CD45RAdepleted PBMC,after G-CSFstim
2,p144852,HD,pos,2705pos,neg,ch45-2,PBMC,after G-CSFstim
3,p1004,HD,pos,2705pos,neg,child_Leu,PBMC,after G-CSFstim
4,p112635,HD,pos,neg,2702pos,ch45-2,PBMC,after G-CSFstim
5,p434,HD,neg,27:05neg,NaN,child_Leu,PBMC,after G-CSFstim
6,p556,HD,neg,27:05neg,NaN,child_Leu,PBMC,after G-CSFstim
7,p744,HD,neg,27:05neg,NaN,child_Leu,PBMC,after G-CSFstim
8,p754,HD,neg,27:05neg,NaN,child_Leu,PBMC,after G-CSFstim
9,p1005,HD,neg,27:05neg,NaN,child_Leu,PBMC,after G-CSFstim


In [105]:
joint = pd.concat([meta2.rename(
    columns={'id': 'donor_id', 'state': 'disease_status', 'b27status': 'b27'}).replace({'HD': 'hd'}), 
           meta3.rename(columns={'id': 'donor_id', 'state': 'disease_status'}).replace({'h': 'hd'})])
joint

,donor_id,disease_status,b27,B2705,B27other,proj,sample_type,time_point
0,Koc,hd,pos,2705pos,neg,GCSF,PBMC,normal blood
1,Make,hd,pos,2705pos,neg,ch45,CD45RAdepleted PBMC,after G-CSFstim
2,p144852,hd,pos,2705pos,neg,ch45-2,PBMC,after G-CSFstim
3,p1004,hd,pos,2705pos,neg,child_Leu,PBMC,after G-CSFstim
4,p112635,hd,pos,neg,2702pos,ch45-2,PBMC,after G-CSFstim
5,p434,hd,neg,27:05neg,NaN,child_Leu,PBMC,after G-CSFstim
6,p556,hd,neg,27:05neg,NaN,child_Leu,PBMC,after G-CSFstim
7,p744,hd,neg,27:05neg,NaN,child_Leu,PBMC,after G-CSFstim
8,p754,hd,neg,27:05neg,NaN,child_Leu,PBMC,after G-CSFstim
9,p1005,hd,neg,27:05neg,NaN,child_Leu,PBMC,after G-CSFstim


In [106]:
joint = pd.concat([joint, meta1.sort_values(by='donor_id').rename(columns={'df_name': 'sample_name', 
                                                 'b27_status': 'b27', 
                                                 'source': 'sample_type'}).replace(
    {'+': 'pos', '-': 'neg', 'PB': 'PBMC'})]).fillna('nan')

In [107]:
joint.donor_id.nunique()

60

In [108]:
joint.disease_status.value_counts()

disease_status
as    35
hd    33
Name: count, dtype: int64

In [109]:
joint.b27.value_counts()

b27
pos    44
neg    24
Name: count, dtype: int64

In [110]:
joint = joint.drop(columns=['B2705', 'B27other'])

In [111]:
joint

,donor_id,disease_status,b27,proj,sample_type,time_point,sample_name,fraction
0,Koc,hd,pos,GCSF,PBMC,normal blood,nan,nan
1,Make,hd,pos,ch45,CD45RAdepleted PBMC,after G-CSFstim,nan,nan
2,p144852,hd,pos,ch45-2,PBMC,after G-CSFstim,nan,nan
3,p1004,hd,pos,child_Leu,PBMC,after G-CSFstim,nan,nan
4,p112635,hd,pos,ch45-2,PBMC,after G-CSFstim,nan,nan
...,...,...,...,...,...,...,...,...
22,Vol,as,pos,nan,PBMC,nan,as_Vol_PB_F,bulk
33,ZN,hd,pos,nan,PBMC,nan,hd_ZN_PB_F,bulk
23,Zakh,as,pos,nan,PBMC,nan,as_Zakh_PB_F,bulk
32,Zbot,hd,pos,nan,PBMC,nan,hd_Zbot_PB_F,bulk


In [112]:
joint['sample_name'] = joint.apply(lambda x: x['sample_name'] if x.sample_name != 'nan' else x.donor_id, axis=1)

In [113]:
joint

,donor_id,disease_status,b27,proj,sample_type,time_point,sample_name,fraction
0,Koc,hd,pos,GCSF,PBMC,normal blood,Koc,nan
1,Make,hd,pos,ch45,CD45RAdepleted PBMC,after G-CSFstim,Make,nan
2,p144852,hd,pos,ch45-2,PBMC,after G-CSFstim,p144852,nan
3,p1004,hd,pos,child_Leu,PBMC,after G-CSFstim,p1004,nan
4,p112635,hd,pos,ch45-2,PBMC,after G-CSFstim,p112635,nan
...,...,...,...,...,...,...,...,...
22,Vol,as,pos,nan,PBMC,nan,as_Vol_PB_F,bulk
33,ZN,hd,pos,nan,PBMC,nan,hd_ZN_PB_F,bulk
23,Zakh,as,pos,nan,PBMC,nan,as_Zakh_PB_F,bulk
32,Zbot,hd,pos,nan,PBMC,nan,hd_Zbot_PB_F,bulk


In [89]:
joint.to_csv( '/projects/immunestatus/rheum/airr_format/metadata.tsv', sep='\t', index=False)